# NASA POWER&mdash;Weather Station Comparison

This notebook queries several of the NASA POWER datasets to return available parameters, and plot temperature at 2 and 10 meters at a specific location over a one year period.

1) Load dependencies

In [ ]:
import zarr
from zarr.storage import MemoryStore, FsspecStore
from zarr import Group
import numpy as np
import pandas as pd
from zarr.experimental.cache_store import CacheStore
import matplotlib.pyplot as plt

2) Location and time range

Our site is only ~100 m x ~150 m, so it is most likely covered by a single satellite pixel. We'll collect data for the northwest corner.

In [ ]:
sample_location = [
    { "latitude": 46.251788, "longitude": -119.728785 },
    { "latitude": 46.251790, "longitude": -119.728371 },
    { "latitude": 46.251407, "longitude": -119.728780 },
    { "latitude": 46.251414, "longitude": -119.728369 },
]

start_date = pd.Timestamp("2023-05-01")
end_date = pd.Timestamp("2024-06-01")

3) Cached Zarr store

Create a function that returns a Zarr group based on a provided path using a cached store. Caching improves performance when repeatedly accessing array subsections.

In [ ]:
def load_store(zarr_path: str) -> Group:
    """Load Zarr store from S3 with caching."""
    s3_store = FsspecStore.from_url(
        zarr_path,
        read_only=True,
        storage_options={"anon": True},
    )
    cache_store = CacheStore(
        store=s3_store,
        cache_store=MemoryStore(),
        max_size=1024 * 1024 * 1024,  # 1 GB cache
    )
    return zarr.open_group(store=cache_store, mode='r')

4) Extract array data

Create a function that returns a Pandas DataFrame containing array data over a specified spatiotemporal range

In [ ]:
def extract_array_slice(group: Group, array_names: list[str], lat: float, lon: float, start_date: pd.Timestamp, end_date: pd.Timestamp) -> pd.DataFrame:
    """Extract a slice of the array for the given polygon and date
    range."""
 
    latitudes = group['lat'][:]
    longitudes = group['lon'][:]
    time_units = group['time'].attrs['units']
    time_base_unit = time_units.split('since')[0].strip()
    time_origin_string = ' '.join(w.strip() for w in time_units.split('since')[1:])
    origin_time = pd.to_datetime(time_origin_string)
    times = None
    if time_base_unit == 'days':
        times = pd.to_datetime(group['time'][:], unit='D', origin=origin_time)
    elif time_base_unit == 'hours':
        times = pd.to_datetime(group['time'][:], unit='h', origin=origin_time)
    else:
        raise ValueError(f"Unsupported time base unit: {time_base_unit!r} in time units {time_units!r}")
 
    # Get arrays of interest
    arrays = {name: group[name] for name in array_names}
 
    # Find nearest indices for latitude and longitude
    lat_idx = np.abs(latitudes - lat).argmin()
    lon_idx = np.abs(longitudes - lon).argmin()
 
    # Find time indices within the date range
    time_indices = np.where((times >= start_date) & (times <= end_date))[0]
    selected_times = times[time_indices]
 
    # Extract data slice
    data_slices = {name: array[time_indices, lat_idx, lon_idx] for name, array in arrays.items()}
    units = {name: array.attrs.get('units', 'unknown') for name, array in arrays.items()}

    print(f"Extracted array slices for:")
    for name in array_names:
        print(f"  - {name} (units: {units[name]}) at lat: {latitudes[lat_idx]}, lon: {longitudes[lon_idx]}")

    # Create DataFrame
    return pd.DataFrame({
        'date': selected_times,
        **{name: data_slices[name] for name in array_names},
    })

5) List dataset parameters

List the parameters of the hourly data available from NASA POWER.

In [ ]:
data_paths = {
    "flashflux (daily LST)": "s3://nasa-power/flashflux/spatial/power_flashflux_daily_spatial_lst.zarr",
    "geosit (hourly UTC)": "s3://nasa-power/geosit/spatial/power_geosit_hourly_spatial_utc.zarr",
    "merra2 (hourly UTC)": "s3://nasa-power/merra2/spatial/power_merra2_hourly_spatial_utc.zarr",
    "syn1deg (hourly UTC)": "s3://nasa-power/syn1deg/spatial/power_syn1deg_hourly_spatial_utc.zarr",
}


for dataset_name, path in data_paths.items():
    print("========================================")
    store = load_store(path)
    print(f"Variables in {dataset_name}:")
    for name, array in store.arrays():
        print(f"- {name}: {array.metadata.attributes['long_name']} [{array.metadata.attributes['units']}]")

6) Temperature data

Extract temperature at 2 and 10 m from the MERRA2 dataset. (This step can take a few minutes.)

In [ ]:
parameters = ['T2M', 'T10M']  # 2-meter and 10-meter temperature

# Use the northwest corner of the sample location for querying the data
lat = sample_location[0]['latitude']
lon = sample_location[0]['longitude']
store = load_store(data_paths["merra2 (hourly UTC)"])
temperature_data = extract_array_slice(store, parameters, lat, lon, start_date, end_date)

7) Plot temperature

Plot the temperature at 2 and 10 m over the measurement range.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(temperature_data['date'], temperature_data['T2M'], label='T2M', marker='o')
plt.plot(temperature_data['date'], temperature_data['T10M'], label='T10M', marker='x')
plt.title(f'Temperature Data at Lat: {lat}, Lon: {lon}')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid()
plt.show()